# SCNN on TuSimple — Google Colab

Trains SCNN lane detection end to end on a Colab GPU, driving the repo's own
`scnn_tusimple.py` rather than re-implementing the loop, so what runs here is the
same code path that runs on a workstation.

**Enable the GPU first:** Runtime → Change runtime type → GPU (A100 if you have it).

At the reference schedule (`batch 32 x 1500 iterations` = 48,000 images, matching
`experiments/exp0`): ~15-25 min on an A100, longer on a T4 or V100 — the batch
size adapts to whatever card you land on.

Order: runtime -> clone -> data -> smoke test -> train -> curves -> score ->
visualise -> save.

## 1. Runtime

In [ ]:
import os, pathlib

try:
    from google.colab import userdata
except ImportError as e:
    raise SystemExit("This notebook targets Google Colab.") from e

WORK      = "/content"
REPO_DIR  = f"{WORK}/SCNN_Culane"
DATA_ROOT = f"{WORK}/TUSimple"
EXP_DIR   = f"{WORK}/exp_tusimple"
print("work dir :", WORK)
print("data     :", DATA_ROOT)
print("exp      :", EXP_DIR)

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU"
free, total = (v / 2**30 for v in torch.cuda.mem_get_info())
name = torch.cuda.get_device_name(0)
print(f"torch {torch.__version__}")
print(f"{name}: {total:.1f} GiB total, {free:.1f} GiB free")
if total - free > 0.5:
    print("warning: something already holds GPU memory; batch size will be scaled down")
if "A100" not in name:
    print(f"note: {name} rather than an A100 - batch adapts, expect a longer run")

## 2. Repository and dependencies

Colab ships torch/torchvision/numpy/matplotlib. Only the four extras are needed:
`opencv-python`, `scikit-learn` (the TuSimple evaluator imports it), `tqdm`, `gdown`.

In [ ]:
REPO_URL = "https://github.com/harshwadhawe/SCNN_Culane.git"

if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git pull -q
!git log --oneline -1

In [ ]:
!pip install -q opencv-python scikit-learn tqdm gdown -U kagglehub

In [ ]:
import torch, cv2, sklearn, numpy
print("cv2", cv2.__version__, "| sklearn", sklearn.__version__, "| numpy", numpy.__version__)
print("cuda:", torch.cuda.is_available())

## 3. Dataset

**Fetch once, cache on Drive, restore every session after.**

The Kaggle archive is ~13 GB because it ships all 20 frames of every clip, but
only frame `20.jpg` is annotated — 6,408 images. Exporting just those, plus the
labels and the generated `seg_label/`, gives ~2 GB: small enough for Drive, and
it restores in a couple of minutes instead of re-downloading 13 GB.

The cells below are idempotent. Run them every session: the first run builds the
cache, later runs skip straight to the restore.

Note it stays **zipped on Drive and unzips to `/content`**. Training reads all
6,408 JPEGs every epoch, and serving those over the Drive mount would dominate
runtime — one big file copies far faster than thousands of small ones.

### 3a. Mount Drive and look for the cache

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR  = pathlib.Path("/content/drive/MyDrive/scnn")
CACHE_ZIP  = DRIVE_DIR / "tusimple_slim.zip"
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CACHED = CACHE_ZIP.exists()
print(f"cache: {CACHE_ZIP}")
print(f"found: {CACHED}" + (f"  ({CACHE_ZIP.stat().st_size/2**30:.2f} GiB)" if CACHED else "  -> will build it"))

### 3b. First run only — fetch from Kaggle and build the cache

Skipped entirely once the cache exists. Needs one Colab secret (🔑 in the left
sidebar) named **`KAGGLE_API_TOKEN`**, notebook access enabled, holding the token
from kaggle.com → Settings → *API*. `kagglehub` authenticates from that token
alone — no username, no key, no `kaggle.json`.

In [ ]:
if not CACHED:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    from kagglehub.config import get_kaggle_credentials
    assert get_kaggle_credentials().api_key, "KAGGLE_API_TOKEN is empty"
    print("authenticated from KAGGLE_API_TOKEN")

In [ ]:
if not CACHED:
    import kagglehub
    SRC_DIR = kagglehub.dataset_download("manideep1108/tusimple")   # ~13 GB
    print("downloaded to:", SRC_DIR)

`--link-from` builds the layout `dataset/Tusimple.py` expects out of one symlink
per clip directory, so the 13 GB cache is not duplicated. `--export` then copies
only the referenced frames into a standalone tree — real files, not links.

In [ ]:
if not CACHED:
    !python download_tusimple.py --dest /content/TUSimple --link-from $SRC_DIR

Generating `seg_label/` now means the cache carries it, and no later session pays
the ~2 min label-generation cost. This is the slow cell of the first run.

In [ ]:
if not CACHED:
    !python -c "import dataset; dataset.Tusimple('/content/TUSimple', 'train', None)"

In [ ]:
if not CACHED:
    !python download_tusimple.py --dest /content/TUSimple --export /content/tusimple_slim

Stored, not deflated (`-0`): JPEG and PNG are already compressed, so this only
costs CPU otherwise. Copying to Drive is the slow part — expect a few minutes.

In [ ]:
if not CACHED:
    !cd /content && zip -0 -qr tusimple_slim.zip tusimple_slim
    !ls -la /content/tusimple_slim.zip

In [ ]:
if not CACHED:
    import shutil, time
    t0 = time.time()
    shutil.copyfile("/content/tusimple_slim.zip", CACHE_ZIP)
    print(f"cached to Drive in {time.time()-t0:.0f}s: {CACHE_ZIP.stat().st_size/2**30:.2f} GiB")

### 3c. Restore into the session

In [ ]:
DATA_ROOT = "/content/tusimple_slim"

if not pathlib.Path(DATA_ROOT).is_dir():
    import time
    t0 = time.time()
    !unzip -q -o $CACHE_ZIP -d /content
    print(f"restored in {time.time()-t0:.0f}s")
else:
    print("already unpacked in this session")

In [ ]:
!python download_tusimple.py --dest $DATA_ROOT --verify-only

All four lines should read `ok` with **0 images missing**:
`label_data_0313.json` 2858, `label_data_0531.json` 358, `label_data_0601.json`
410, `test_label.json` 2782 — 3268 train / 358 val / 2782 test.

From here on nothing touches Kaggle or Drive. After a disconnect, re-run from 3a:
the cache is found, 3b is skipped, and you are training again in a couple of
minutes.

## 4. Smoke test

Twenty iterations, no evaluation. Confirms CUDA, the dataloader, AMP and
checkpointing all work before committing to the real run.

**First run is slow to start:** `dataset/Tusimple.py` generates `seg_label/` for
all 6408 frames on first instantiation — a minute or two with no output. It is
cached afterwards.

In [ ]:
!python scnn_tusimple.py --data $DATA_ROOT --exp-dir $WORK/smoke --max-iter 20 --no-eval

## 5. Train

Defaults resolve from the hardware: on a 40 GB A100 that is **batch 32**, which
is exactly the `experiments/exp0` reference, so `lr` lands on 0.15 and
`max_iter 1500` sees the same 48,000 images as the published run.

`max_iter` counts **iterations, not images**. If the batch comes out smaller than
32 (a busy GPU, or a T4/V100), scale `max_iter` by `32/batch` to keep the same
number of images seen — otherwise you silently train on less data.

Colab drops idle sessions. The script checkpoints every epoch and `--resume`
continues from `latest.pth`, so re-running the cell after a disconnect resumes
rather than restarting.

In [ ]:
!python scnn_tusimple.py --data $DATA_ROOT --exp-dir $EXP_DIR --max-iter 1500

After a disconnect, run this instead of the cell above:

In [ ]:
# !python scnn_tusimple.py --data $DATA_ROOT --exp-dir $EXP_DIR --max-iter 1500 --resume

## 6. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

hist = pd.read_csv(f"{EXP_DIR}/history.csv")
hist.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), layout="constrained")
axes[0].plot(hist.epoch, hist.train_loss, "o-", lw=1.4)
axes[0].set_xlabel("epoch"); axes[0].set_title("train loss")

axes[1].plot(hist.epoch, hist.val_loss, "o-", lw=1.4, color="tab:red")
axes[1].set_xlabel("epoch"); axes[1].set_title("val loss")

axes[2].plot(hist.epoch, hist.lr, lw=1.6, color="tab:green")
axes[2].set_xlabel("epoch"); axes[2].set_title("PolyLR")
for ax in axes:
    ax.grid(alpha=.25)
plt.show()
print(f"best val {hist.val_loss.min():.4f} at epoch {int(hist.val_loss.idxmin())} "
      f"| {hist.elapsed_s.iloc[-1]/60:.1f} min total")

## 7. Score

`LaneEval.bench_one_submit` runs automatically at the end of training. Re-run it
standalone with `--eval-only`, which loads `best.pth` rather than the last epoch.

Reference for `experiments/exp0`: **94.16% accuracy, FP 0.0735, FN 0.0825**.

In [ ]:
import json, pathlib

result = pathlib.Path(f"{EXP_DIR}/evaluation_result.txt")
if not result.exists():
    !python scnn_tusimple.py --data $DATA_ROOT --exp-dir $EXP_DIR --eval-only

In [ ]:
rows = json.loads(pathlib.Path(f"{EXP_DIR}/evaluation_result.txt").read_text())
ref = {"Accuracy": 0.9416, "FP": 0.0735, "FN": 0.0825}
print(f"{'metric':<10}{'this run':>12}{'exp0 ref':>12}")
for r in rows:
    print(f"{r['name']:<10}{r['value']:>12.4f}{ref.get(r['name'], float('nan')):>12.4f}")

## 8. Predictions on test images

In [ ]:
import numpy as np, torch, cv2
from model import SCNN
from utils.transforms import Compose, Resize, ToTensor, Normalize
from utils.prob2lines import getLane
import dataset

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
RESIZE = (512, 288)
device = torch.device("cuda")

net = SCNN(input_size=RESIZE, pretrained=False)
net.load_state_dict(torch.load(f"{EXP_DIR}/best.pth", map_location="cpu", weights_only=False)["net"])
net = net.to(device).eval()
print("loaded best.pth")

In [ ]:
# the repo's colour arrays are BGR (they end at cv2.imwrite); matplotlib draws RGB
LANE_COLORS = np.array([[255, 125, 0], [0, 255, 0], [0, 0, 255], [0, 255, 255]], np.uint8)[:, ::-1]

def overlay(rgb, mask, alpha=.8):
    lane = np.zeros_like(rgb)
    for i in range(4):
        lane[mask == i + 1] = LANE_COLORS[i]
    return cv2.addWeighted(lane, alpha, rgb, 1., 0.)

def predicted_mask(seg, exist, thresh=.5):
    mask = np.argmax(seg, axis=0)
    for i in range(4):
        if exist[i] <= thresh:
            mask[mask == i + 1] = 0
    return mask

In [ ]:
test_set = dataset.Tusimple(DATA_ROOT, "test",
                            Compose(Resize(RESIZE), ToTensor(), Normalize(MEAN, STD)))
picks = range(0, len(test_set), max(1, len(test_set) // 6))
batch = dataset.Tusimple.collate([test_set[i] for i in list(picks)[:6]])

with torch.no_grad():
    seg, exist = net(batch["img"].to(device))[:2]
    seg = torch.softmax(seg.float(), 1).cpu().numpy()
    exist = exist.float().cpu().numpy()
print("predicted on", len(seg), "test frames")

In [ ]:
import matplotlib.pyplot as plt

n = len(seg)
fig, axes = plt.subplots(n, 1, figsize=(9, 2.6 * n), layout="constrained")
for j, ax in enumerate(np.atleast_1d(axes)):
    rgb = cv2.cvtColor(cv2.imread(batch["img_name"][j]), cv2.COLOR_BGR2RGB)
    rgb = Resize(RESIZE)({"img": rgb})["img"]
    ax.imshow(overlay(rgb, predicted_mask(seg[j], exist[j])))
    ax.set_title(f"{'/'.join(batch['img_name'][j].split('/')[-3:])}   "
                 f"exist {(exist[j] > .5).astype(int).tolist()}", fontsize=8)
    ax.axis("off")
plt.show()

### Lane coordinates, as the evaluator sees them

In [ ]:
flags = [int(exist[0, k] > .5) for k in range(4)]
lanes = getLane.prob2lines_tusimple(seg[0], flags, resize_shape=(720, 1280), y_px_gap=10, pts=56)
print(f"{len(lanes)} lanes on {batch['img_name'][0].split('/')[-3:]}")
for i, l in enumerate(lanes):
    xs = [int(x) for x, _ in l if x > 0]
    print(f"  lane {i+1}: {len(xs)} points, x {xs[0] if xs else '-'} -> {xs[-1] if xs else '-'}")

## 9. Save to Drive

`/content` is wiped when the runtime recycles. `best.pth` is ~164 MB.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = "/content/drive/MyDrive/scnn_tusimple"
!mkdir -p $DEST
!cp $EXP_DIR/best.pth $EXP_DIR/history.csv $EXP_DIR/train.log $DEST/ 2>/dev/null
!cp $EXP_DIR/cfg.json $EXP_DIR/evaluation_result.txt $DEST/ 2>/dev/null
!ls -la $DEST

## Notes

- **The Drive cache is the point.** Section 3b runs once; every session after
  restores ~2 GB from `MyDrive/scnn/tusimple_slim.zip` instead of pulling 13 GB
  from Kaggle. To rebuild it, delete that file and re-run section 3.
- **Unzip to `/content`, never train off the Drive mount.** Every epoch reads all
  6,408 JPEGs; over Drive that dominates runtime.
- **The cache carries `seg_label/`,** so no session pays the label-generation cost
  after the first.
- **`max_iter` counts iterations, not images.** Change the batch size and scale it
  by `32/batch` or you train on proportionally less data.
- **Disconnects.** Re-run from 3a: the cache is found, 3b is skipped, and
  `--resume` continues training from `latest.pth`.
- **Kaggle terms.** Open the dataset page in a browser once; the API 403s until
  they are accepted.
- **CULane next.** `download_culane.py` fetches it, but at ~55 GB it will not fit
  in a Colab session's disk -- use a subset, or run it on a workstation.